# Ablation Study — AsymmetricTriNet
Runs all 7 modality-combination variants (3 single + 3 dual + 1 full).
## Steps
1. Mount Drive and set paths
2. Copy `ablation_trinet.py` into `python/src/models/`
3. Run training
4. Run evaluation and print table

In [18]:
from google.colab import drive
from pathlib import Path
import sys, os

drive.mount('/content/drive', force_remount=True)

PROJECT_ROOT = Path('/content/drive/Othercomputers/My Laptop/thesis_project')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
print('Working directory:', os.getcwd())

Mounted at /content/drive
Working directory: /content/drive/Othercomputers/My Laptop/thesis_project


In [ ]:
# ── Step 1: Copy ablation_trinet.py into your models folder ────────────────
# Upload ablation_trinet.py to Colab first (Files panel), then run this cell.
# OR just paste the class definition directly below.

import shutil
src = Path('/content/ablation_trinet.py')          # uploaded location
dst = PROJECT_ROOT / 'python/src/models/ablation_trinet.py'

if src.exists():
    shutil.copy(src, dst)
    print(f'Copied: {dst}')
elif dst.exists():
    print(f'Already in place: {dst}')
else:
    print('WARNING: ablation_trinet.py not found. Paste the class manually or upload.')

Already in place: /content/drive/Othercomputers/My Laptop/thesis_project/python/src/models/ablation_trinet.py


In [16]:
# ── Step 2: Run Training ────────────────────────────────────────────────────
# This will sequentially train all 7 variants.
# On Colab L4 GPU: expect ~15-20 min per variant = ~2 hours total.
# Already-trained variants are skipped automatically.

%run scripts/run_ablation.py

Device: cuda

Loading dataset...
  Train: 20000 samples
  Val  : 5000 samples
  Test : 25000 samples

  Training: STFT only
  Disabled branches: {1, 2}
  Epoch 01/50 | LR: 0.000999 | Train: 2.4710 (CE 2.1510 / SupCon 3.1995) | Val: 1.8459 | Val Acc: 43.20%
  Epoch 05/50 | LR: 0.000976 | Train: 1.5874 (CE 1.3510 / SupCon 2.3640) | Val: 0.9725 | Val Acc: 75.86%
  Epoch 10/50 | LR: 0.000905 | Train: 1.5580 (CE 1.3261 / SupCon 2.3190) | Val: 0.8005 | Val Acc: 85.82%
  Epoch 15/50 | LR: 0.000794 | Train: 1.4715 (CE 1.2498 / SupCon 2.2168) | Val: 0.8245 | Val Acc: 85.44%
  Epoch 20/50 | LR: 0.000655 | Train: 1.4732 (CE 1.2524 / SupCon 2.2079) | Val: 0.8020 | Val Acc: 84.76%
  Epoch 25/50 | LR: 0.000500 | Train: 1.4064 (CE 1.1945 / SupCon 2.1189) | Val: 0.7971 | Val Acc: 85.52%
  Epoch 30/50 | LR: 0.000345 | Train: 1.4104 (CE 1.2008 / SupCon 2.0960) | Val: 0.8107 | Val Acc: 87.48%
  Epoch 35/50 | LR: 0.000206 | Train: 1.3479 (CE 1.1461 / SupCon 2.0182) | Val: 0.8907 | Val Acc: 85.86%
  Epoch 

In [19]:
# ── Step 3: Run Evaluation and print the ablation table ────────────────────

%run scripts/ablation_eval.py

Device: cuda

Loading eval datasets...
  Loaded 13 datasets.

Evaluating: STFT only
  SNR  -14 dB → 42.38%
  SNR  -12 dB → 61.54%
  SNR  -10 dB → 80.08%
  SNR   -8 dB → 88.12%
  SNR   -6 dB → 91.58%
  SNR   -4 dB → 94.14%
  SNR   -2 dB → 96.00%
  SNR   +0 dB → 96.90%
  SNR   +2 dB → 98.30%
  SNR   +4 dB → 99.00%
  SNR   +6 dB → 99.26%
  SNR   +8 dB → 99.28%
  SNR  +10 dB → 99.40%

Evaluating: IQ only
  SNR  -14 dB → 24.00%
  SNR  -12 dB → 35.16%
  SNR  -10 dB → 53.34%
  SNR   -8 dB → 71.86%
  SNR   -6 dB → 85.16%
  SNR   -4 dB → 92.34%
  SNR   -2 dB → 95.86%
  SNR   +0 dB → 97.82%
  SNR   +2 dB → 98.46%
  SNR   +4 dB → 98.34%
  SNR   +6 dB → 99.02%
  SNR   +8 dB → 98.92%
  SNR  +10 dB → 99.04%

Evaluating: IF only
  SNR  -14 dB → 15.28%
  SNR  -12 dB → 19.94%
  SNR  -10 dB → 29.80%
  SNR   -8 dB → 43.16%
  SNR   -6 dB → 58.64%
  SNR   -4 dB → 73.74%
  SNR   -2 dB → 84.80%
  SNR   +0 dB → 91.08%
  SNR   +2 dB → 94.56%
  SNR   +4 dB → 94.82%
  SNR   +6 dB → 95.80%
  SNR   +8 dB → 95.50%


In [ ]:
# ── Optional: Train a SINGLE variant only ──────────────────────────────────
# Useful if one job crashed or timed out.

from python.src.models.ablation_trinet import AblationTriNet

# Example: retrain STFT+IQ only
# Just delete its checkpoint first:
# ckpt = PROJECT_ROOT / 'artifacts/checkpoints/asymmetric_trinet_ablation_stft_iq_seed55_n2500.pt'
# ckpt.unlink(missing_ok=True)
# Then re-run cell above, or call train_variant() directly.

# Quick sanity check — instantiate each variant and count parameters
import torch

variants = [
    ('STFT only',         {1, 2}),
    ('IQ only',           {0, 2}),
    ('IF only',           {0, 1}),
    ('STFT + IQ',         {2}),
    ('STFT + IF',         {1}),
    ('IQ + IF',           {0}),
    ('Full (STFT+IQ+IF)', set()),
]

print(f'{'Variant':<25} {'Params':>12}  Disabled')
print('-' * 55)
for label, disabled in variants:
    m = AblationTriNet(num_classes=10, disabled_branches=disabled)
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'{label:<25} {n:>12,}  {disabled or "none"}')